# YOLOv10n — Xe tự hành miniature (320×240, low-light) 
> **Export**: `.pt` · `.onnx` · `NCNN` (ARM/C++) · `TFLite INT8` (Edge AI / RPi)

| ID | Class | Vật thể thực tế |
|----|-------|-----------------|
| 0 | `stop_sign` | Biển STOP bát giác đỏ thu nhỏ |
| 1 | `no_entry` | Biển cấm (tròn đỏ, vạch trắng) |
| 2 | `red_light` | LED đỏ trên cột đèn |
| 3 | `yellow_light` | LED vàng |
| 4 | `green_light` | LED xanh |
| 5 | `person` | Biển/hình người đi bộ (zebra crossing) |

> **Dataset**: ảnh 320×240, tối, chụp từ camera xe.  
```
/dataset/
├── images/
│   ├── train/  *.jpg
│   └── val/    *.jpg
└── labels/
    ├── train/  *.txt
    └── val/    *.txt
```

## ⚙️ Cell 1 — Cấu hình (chỉnh tại đây)

In [ ]:
from pathlib import Path

# ─────────────────────────────────────────────────────────────────
#  DATASET
# ─────────────────────────────────────────────────────────────────
DATASET_INPUT = "/kaggle/input/datasets/buiminhphuongx/dataset/dataset"

# ─────────────────────────────────────────────────────────────────
#  MODEL  ← YOLOv8n → YOLOv10n: NMS-free, params ít hơn 28%
# ─────────────────────────────────────────────────────────────────
MODEL      = "yolov10n"   # đổi sang "yolov10s" nếu GPU đủ mạnh

# ─────────────────────────────────────────────────────────────────
#  TRAINING  (giữ nguyên hyperparams đã tune tốt từ v8n)
# ─────────────────────────────────────────────────────────────────
EPOCHS     = 100
IMG_SIZE   = 320
BATCH      = 32
PATIENCE   = 25           # early-stop sau 25 epoch plateau

# ─────────────────────────────────────────────────────────────────
#  CLASSES
# ─────────────────────────────────────────────────────────────────
CLASSES = ["stop_sign", "no_entry", "red_light", "yellow_light", "green_light", "person"]

# ─────────────────────────────────────────────────────────────────
#  PATHS
# ─────────────────────────────────────────────────────────────────
WORK       = Path("/kaggle/working")
DATASET    = WORK / "dataset"
RUNS       = WORK / "runs"
OUT        = WORK / "output"
OUT.mkdir(parents=True, exist_ok=True)

print(f"Model    : {MODEL}  (YOLOv10n — NMS-free)")
print(f"ImgSize  : {IMG_SIZE}×{IMG_SIZE}")
print(f"Epochs   : {EPOCHS}  |  Patience: {PATIENCE}")
print(f"Batch    : {BATCH}")
print(f"Classes  : {CLASSES}")
print(f"Dataset  : {DATASET_INPUT}")

## 1. Cài đặt

In [ ]:
%%capture
import subprocess, sys

# ultralytics>=8.3 hỗ trợ YOLOv10; onnx + onnxsim cho export ONNX/NCNN
# KHÔNG cài onnx2tf hay tensorflow ở đây — Kaggle đã có TF 2.19 sẵn
subprocess.run([sys.executable, "-m", "pip", "install",
                "ultralytics>=8.3",
                "onnx>=1.14", "onnxsim",
                "-q"], check=True)

import torch, cv2, ultralytics
print(f"ultralytics {ultralytics.__version__}  |  "
      f"torch {torch.__version__}  |  "
      f"opencv {cv2.__version__}")

if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f"GPU: {g.name}  |  VRAM: {g.total_memory/1e9:.1f} GB")
    DEVICE = "0"
else:
    print("Không có GPU — bật Accelerator trong Settings!")
    DEVICE = "cpu"

from ultralytics import YOLO
print("YOLOv10n sẵn sàng")


## 2. Chuẩn bị Dataset

In [ ]:
import shutil, yaml
from pathlib import Path

src = Path(DATASET_INPUT)
if not src.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {src}\n"
        "→ Trên Kaggle: nhấn '+ Add data' → chọn dataset của bạn"
    )

# Copy sang /kaggle/working để tránh read-only
if DATASET.exists():
    shutil.rmtree(DATASET)
shutil.copytree(str(src), str(DATASET))

# Thống kê nhanh
print("Dataset structure:")
for split in ["train", "val"]:
    imgs = list((DATASET/"images"/split).glob("*.[jp][pn]g"))
    lbls = list((DATASET/"labels"/split).glob("*.txt"))
    counts = [0] * len(CLASSES)
    for lf in lbls:
        for line in lf.read_text().splitlines():
            p = line.strip().split()
            if len(p) == 5:
                cid = int(p[0])
                if 0 <= cid < len(CLASSES):
                    counts[cid] += 1
    print(f"  [{split}]  {len(imgs)} ảnh  |  {sum(counts)} boxes")
    for i, (cls, cnt) in enumerate(zip(CLASSES, counts)):
        bar = "█" * min(cnt // max(sum(counts)//30, 1), 25)
        print(f"    [{i}] {cls:<14} {cnt:4d}  {bar}")

# Tạo dataset.yaml
yaml_path = DATASET / "dataset.yaml"
yaml.dump({
    "path"  : str(DATASET.resolve()),
    "train" : "images/train",
    "val"   : "images/val",
    "nc"    : len(CLASSES),
    "names" : CLASSES,
}, open(yaml_path, "w"), default_flow_style=False, allow_unicode=True)
print(f"\ndataset.yaml → {yaml_path}")

## 3. Xem trước ảnh & nhãn

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches

COLORS = ["#e74c3c","#c0392b","#e67e22","#f1c40f","#2ecc71","#3498db"]

def show_labels(img_path, lbl_path, ax):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img); ax.set_title(img_path.name, fontsize=8); ax.axis("off")
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            p = line.strip().split()
            if len(p) != 5: continue
            cid, cx, cy, bw, bh = int(p[0]), *map(float, p[1:])
            x1 = (cx - bw/2)*w; y1 = (cy - bh/2)*h
            col = COLORS[cid % len(COLORS)]
            ax.add_patch(mpatches.Rectangle((x1,y1), bw*w, bh*h,
                lw=2, edgecolor=col, facecolor="none"))
            ax.text(x1, max(0,y1-2), CLASSES[cid], fontsize=7, color=col,
                    fontweight="bold", bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.7))

imgs = sorted((DATASET/"images"/"train").glob("*.[jp][pn]g"))[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, ip in zip(axes.flat, imgs):
    show_labels(ip, DATASET/"labels"/"train"/(ip.stem+".txt"), ax)
for ax in axes.flat[len(imgs):]: ax.axis("off")
plt.suptitle("Train set — ảnh thực tế + annotation", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

## 4. Train YOLOv10n

In [ ]:
from ultralytics import YOLO

# YOLOv10n — NMS-free (dual-label assignment khi train, one-to-one khi infer)
# Toàn bộ hyperparameter giữ nguyên từ v8n (đã tune tốt cho ảnh tối 320×240)
model = YOLO(f"{MODEL}.pt")

results = model.train(
    data    = str(yaml_path),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH,
    device  = DEVICE,
    project = str(RUNS),
    name    = "traffic",
    exist_ok= True,

    # ── Augmentation cho ảnh tối 320×240 ─────────────────────────────
    hsv_v       = 0.6,       # jitter brightness ±60% — ảnh đường đua rất tối
    hsv_s       = 0.5,
    hsv_h       = 0.01,      # hue nhẹ — màu LED nhạy cảm
    degrees     = 3.0,
    translate   = 0.05,
    scale       = 0.4,       # object nhỏ → scale mạnh để học đa dạng size
    shear       = 1.0,
    perspective = 0.0003,
    flipud      = 0.0,       # không lật dọc
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.05,      # nhẹ — tránh làm mờ LED nhỏ
    copy_paste  = 0.0,       # off — background đường đua đồng nhất

    # ── Optimizer ────────────────────────────────────────────────────
    optimizer    = "AdamW",
    lr0          = 0.001,
    lrf          = 0.005,    # lr_final thấp — fine-tune object nhỏ
    weight_decay = 0.0005,
    warmup_epochs= 5,        # warmup dài hơn — dataset nhỏ cần ổn định

    # ── Loss ─────────────────────────────────────────────────────────
    box = 10.0,   # tăng box loss — cần bbox chính xác cho object nhỏ
    cls = 0.3,    # giảm cls loss — chỉ 6 class, ít nhầm lẫn
    dfl = 1.5,

    # ── Misc ─────────────────────────────────────────────────────────
    patience    = PATIENCE,
    save_period = 20,
    plots       = True,
    verbose     = False,
)

# Tìm best.pt
best_pt = RUNS / "traffic" / "weights" / "best.pt"
if not best_pt.exists():
    cands = sorted(RUNS.rglob("best.pt"), key=lambda p: p.stat().st_mtime)
    best_pt = cands[-1] if cands else None

print(f"\n{'✓' if best_pt and best_pt.exists() else '✗'} Best weights: {best_pt}")

## 5. Kết quả & Biểu đồ

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

# ── Metrics validation ────────────────────────────────────────
val_model = YOLO(str(best_pt))
m = val_model.val(data=str(yaml_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)

print("┌──────────────────────────────┐")
print("│      VALIDATION METRICS      │")
print("├──────────────────────────────┤")
print(f"│  mAP@0.5      : {m.box.map50:.4f}        │")
print(f"│  mAP@0.5:0.95 : {m.box.map:.4f}        │")
print(f"│  Precision    : {m.box.mp:.4f}        │")
print(f"│  Recall       : {m.box.mr:.4f}        │")
print("├──────────────────────────────┤")
for i, cls in enumerate(CLASSES):
    try:
        ap = m.box.ap50[i]
        bar = "█"*int(ap*20)
        print(f"│  [{i}]{cls:<13}: {ap:.3f} {bar:<20}│")
    except: pass
print("└──────────────────────────────┘")

# ── Training curves ────────────────────────────────────────────
csv_path = RUNS / "traffic" / "results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path); df.columns = df.columns.str.strip()
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    pairs = [
        ("train/box_loss", "val/box_loss",      "Box Loss",    axes[0,0]),
        ("train/cls_loss", "val/cls_loss",      "Class Loss",  axes[0,1]),
        ("train/dfl_loss", "val/dfl_loss",      "DFL Loss",    axes[0,2]),
        ("metrics/precision(B)", None,           "Precision",   axes[1,0]),
        ("metrics/recall(B)",    None,           "Recall",      axes[1,1]),
        ("metrics/mAP50(B)",     None,           "mAP@0.5",     axes[1,2]),
    ]
    for tc, vc, title, ax in pairs:
        if tc in df: ax.plot(df[tc], label="train", lw=1.8, color="#3498db")
        if vc and vc in df: ax.plot(df[vc], label="val", lw=1.8, ls="--", color="#e74c3c")
        ax.set_title(title, fontweight="bold"); ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle("Training Curves — YOLOv10n", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(OUT/"training_curves.png"), dpi=120, bbox_inches="tight")
    plt.show()

# ── Confusion matrix & PR curve ────────────────────────────────
run_dir = RUNS / "traffic"
for fname, title in [("confusion_matrix_normalized.png","Confusion Matrix"),
                      ("PR_curve.png","PR Curve")]:
    p = run_dir / fname
    if p.exists():
        fig, ax = plt.subplots(figsize=(8,6))
        ax.imshow(plt.imread(str(p))); ax.axis("off")
        ax.set_title(title, fontweight="bold")
        plt.tight_layout(); plt.show()

## 6. Inference trên ảnh thực tế

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

infer = YOLO(str(best_pt))
val_imgs = sorted((DATASET/"images"/"val").glob("*.[jp][pn]g"))[:8]

if val_imgs:
    cols = 4; rows = (len(val_imgs)+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows*4))
    axes = np.array(axes).flatten()
    for i, ip in enumerate(val_imgs):
        r = infer.predict(str(ip), imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]
        plotted = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
        axes[i].imshow(plotted); axes[i].set_title(ip.name, fontsize=8)
        axes[i].axis("off")
    for ax in axes[len(val_imgs):]: ax.axis("off")
    plt.suptitle("Inference — Val Set", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(OUT/"inference_val.png"), dpi=100, bbox_inches="tight")
    plt.show()

## 7. Export — `.pt` · `.onnx` · `NCNN` · `TFLite INT8`

| Format | Mục tiêu | Pipeline | Đặc điểm |
|--------|----------|----------|----------|
| **`.pt`** | PyTorch inference, fine-tune | ultralytics native | FP32 |
| **ONNX** | Laptop/Desktop, OpenCV dnn | ultralytics → onnxsim | FP32, opset 12 |
| **NCNN** | Raspberry Pi / ARM (C++) | ultralytics → ncnn | FP16, không cần runtime nặng |
| **TFLite FP32** | Baseline kiểm tra độ chính xác | SavedModel → TFLiteConverter | Không cài thêm gì |
| **TFLite INT8** | Edge AI, C/C++, RPi, MCU | SavedModel → TFLiteConverter + calibration | Full-int quant, nhỏ ~4× |

> **Tại sao không dùng `onnx2tf`?**  
> `onnx2tf` kéo `numpy>=2.x`, xung đột với `tensorflow 2.19` trên Kaggle (yêu cầu `numpy<2.2`).  
> Giải pháp: dùng `ultralytics export(format="saved_model")` → `tf.lite.TFLiteConverter` built-in.  
> TensorFlow đã có sẵn trên Kaggle — **không cài thêm package nào**.


In [ ]:
import shutil, numpy as np, cv2
from pathlib import Path

exp = YOLO(str(best_pt))

# ════════════════════════════════════════════════════════════════════
#  7-A  PyTorch (.pt) + ONNX
# ════════════════════════════════════════════════════════════════════
print("[1/4] Exporting ONNX ...")
onnx_src = Path(str(exp.export(
    format   = "onnx",
    imgsz    = IMG_SIZE,
    opset    = 12,       # tương thích OpenCV 4.x
    simplify = True,
    dynamic  = False,    # batch=1 fixed cho real-time
    half     = False,    # FP32 — tương thích RPi không GPU
)))
onnx_dst = OUT / "traffic_v10n.onnx"
shutil.copy2(str(onnx_src), str(onnx_dst))
shutil.copy2(str(best_pt),  str(OUT/"best.pt"))
print(f"  ✓ .pt   : best.pt  ({(OUT/'best.pt').stat().st_size/1e6:.2f} MB)")
print(f"  ✓ ONNX  : {onnx_dst.name}  ({onnx_dst.stat().st_size/1e6:.2f} MB)")

# ════════════════════════════════════════════════════════════════════
#  7-B  NCNN  — ARM/C++ không cần runtime nặng (Tencent)
# ════════════════════════════════════════════════════════════════════
print("\n[2/4] Exporting NCNN ...")
ncnn_dir_src = Path(str(exp.export(format="ncnn", imgsz=IMG_SIZE, half=False)))
ncnn_dst = OUT / "ncnn_model"
if ncnn_dst.exists(): shutil.rmtree(str(ncnn_dst))
shutil.copytree(str(ncnn_dir_src), str(ncnn_dst))
param_f = next(ncnn_dst.glob("*.param"), None)
bin_f   = next(ncnn_dst.glob("*.bin"),   None)
if param_f and bin_f:
    print(f"  ✓ NCNN param : {param_f.name}  ({param_f.stat().st_size/1e3:.1f} KB)")
    print(f"  ✓ NCNN bin   : {bin_f.name}  ({bin_f.stat().st_size/1e6:.2f} MB)")

# ════════════════════════════════════════════════════════════════════
#  7-C  TFLite FP32 — dùng ultralytics export → SavedModel → TFLite
#
#  Quy trình:
#    best.pt → ultralytics export(format="saved_model")
#              → /runs/traffic/weights/best_saved_model/
#    TFLiteConverter.from_saved_model() → traffic_v10n_fp32.tflite
#
#  Tại sao KHÔNG dùng ultralytics export(format="tflite")?
#  → Kaggle có bug: undefined symbol libtfkernel_sobol_op.so
#  Tại sao KHÔNG dùng onnx2tf?
#  → onnx2tf kéo numpy>=2.x, xung đột với tensorflow 2.19 (yêu cầu numpy<2.2)
#  → SavedModel built-in dùng TF có sẵn, không cài thêm gì
# ════════════════════════════════════════════════════════════════════
print("\n[3/4] Exporting TFLite FP32 (SavedModel → TFLiteConverter) ...")

# Bước 1: export sang SavedModel (ultralytics built-in, không trigger TF kernel bug)
sm_src = Path(str(exp.export(
    format  = "saved_model",
    imgsz   = IMG_SIZE,
    dynamic = False,
    half    = False,
)))
# ultralytics tạo thư mục: best_saved_model/ hoặc best_web_model/
if not sm_src.exists():
    # fallback: tìm trong thư mục weights
    candidates = list((RUNS/"traffic"/"weights").glob("*saved_model*"))
    sm_src = candidates[0] if candidates else sm_src
print(f"  SavedModel: {sm_src}")

# Bước 2: dùng TF 2.x có sẵn trên Kaggle — không cài thêm
import tensorflow as tf
print(f"  TensorFlow: {tf.__version__}")

converter = tf.lite.TFLiteConverter.from_saved_model(str(sm_src))
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_fp32 = converter.convert()

tflite_fp32_dst = OUT / "traffic_v10n_fp32.tflite"
tflite_fp32_dst.write_bytes(tflite_fp32)
print(f"  ✓ TFLite FP32 : {tflite_fp32_dst.name}  ({tflite_fp32_dst.stat().st_size/1e6:.2f} MB)")

# ════════════════════════════════════════════════════════════════════
#  7-D  TFLite INT8 — full integer quantization với calibration data
#       Input/Output: int8 — tối ưu cho Edge AI / C++ / RPi / MCU
# ════════════════════════════════════════════════════════════════════
print("\n[4/4] Exporting TFLite INT8 (full integer quantization) ...")

# Chuẩn bị representative dataset (ảnh val làm calibration)
val_imgs_calib = sorted((DATASET/"images"/"val").glob("*.[jp][pn]g"))[:100]
print(f"  Calibration: {len(val_imgs_calib)} ảnh val")

def representative_dataset():
    for ip in val_imgs_calib:
        img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype(np.float32) / 255.0
        yield [img[np.newaxis, ...]]   # shape: (1, 320, 320, 3)

converter_int8 = tf.lite.TFLiteConverter.from_saved_model(str(sm_src))
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type  = tf.int8    # input INT8
converter_int8.inference_output_type = tf.int8    # output INT8
tflite_int8 = converter_int8.convert()

tflite_int8_dst = OUT / "traffic_v10n_int8.tflite"
tflite_int8_dst.write_bytes(tflite_int8)
print(f"  ✓ TFLite INT8  : {tflite_int8_dst.name}  ({tflite_int8_dst.stat().st_size/1e6:.2f} MB)")
print(f"  ℹ Coral Edge TPU: edgetpu_compiler {tflite_int8_dst.name}")

# ── Summary ───────────────────────────────────────────────────────
print("\n📋 Tóm tắt output:")
for f in sorted(OUT.iterdir()):
    if f.is_file():
        print(f"  {f.name:<40} {f.stat().st_size/1e6:.3f} MB")


## 8. Đóng gói & Tải về

In [ ]:
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

zip_path = WORK / "traffic_v10n_results.zip"
run_dir  = RUNS / "traffic"

print("🔄 Đang gom và nén các file kết quả...")

with zipfile.ZipFile(str(zip_path), "w", zipfile.ZIP_DEFLATED) as zf:

    # 1. Trọng số mô hình
    pt_out = OUT / "best.pt"
    if pt_out.exists():
        zf.write(str(pt_out), "weights/best.pt")

    if onnx_dst.exists():
        zf.write(str(onnx_dst), "weights/traffic_v10n.onnx")

    # 2. NCNN — ARM/C++ (Tencent)
    param_f = next((OUT/"ncnn_model").glob("*.param"), None) if (OUT/"ncnn_model").exists() else None
    bin_f   = next((OUT/"ncnn_model").glob("*.bin"),   None) if (OUT/"ncnn_model").exists() else None
    if param_f: zf.write(str(param_f), f"ncnn/{param_f.name}")
    if bin_f:   zf.write(str(bin_f),   f"ncnn/{bin_f.name}")

    # 3. TFLite — Edge AI / C++ / RPi
    for tflite_file, arc_name in [
        (OUT/"traffic_v10n_fp32.tflite", "tflite/traffic_v10n_fp32.tflite"),
        (OUT/"traffic_v10n_int8.tflite", "tflite/traffic_v10n_int8.tflite"),
    ]:
        if tflite_file.exists():
            zf.write(str(tflite_file), arc_name)

    # 4. Cấu hình dataset
    if yaml_path.exists():
        zf.write(str(yaml_path), "dataset.yaml")

    # 5. Biểu đồ training & inference
    for p in OUT.glob("*.png"):
        zf.write(str(p), f"plots/{p.name}")
    if run_dir.exists():
        for p in run_dir.glob("*.png"):
            zf.write(str(p), f"plots/{p.name}")

    # 6. CSV metrics
    csv_p = run_dir / "results.csv"
    if csv_p.exists():
        zf.write(str(csv_p), "results.csv")

size_mb = zip_path.stat().st_size / 1e6
print(f"\n📦 Đã tạo: {zip_path.name}  ({size_mb:.1f} MB)")
print()
print("Nội dung ZIP:")
with zipfile.ZipFile(str(zip_path)) as zf:
    for info in sorted(zf.infolist(), key=lambda x: x.filename):
        print(f"  {info.filename:<50} {info.file_size/1e6:.3f} MB")
print()
print("👇 BẤM VÀO LINK BÊN DƯỚI ĐỂ TẢI VỀ MÁY:")
display(FileLink(zip_path.name))

## 9. Dùng các format trên xe tự hành / RPi

### ONNX — OpenCV dnn (Laptop / Desktop)
```python
import cv2, numpy as np

# YOLOv10 output: (1, 300, 6) — đã NMS sẵn, không cần NMS thủ công
CLASSES = ["stop_sign","no_entry","red_light","yellow_light","green_light","person"]
CONF    = 0.45; SIZE = 320

net = cv2.dnn.readNetFromONNX("traffic_v10n.onnx")
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

def detect(frame):
    h, w = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (SIZE,SIZE), swapRB=True, crop=False)
    net.setInput(blob); out = net.forward()           # (1, 300, 6)
    results = []
    for det in out[0]:
        if det[4] < CONF: continue
        cid = int(det[5]); sx,sy = w/SIZE, h/SIZE
        x1,y1,x2,y2 = int(det[0]*sx),int(det[1]*sy),int(det[2]*sx),int(det[3]*sy)
        results.append((cid, float(det[4]), x1,y1,x2,y2))
    return results
```

### NCNN — Raspberry Pi / ARM (C++ / Python binding)
```python
import ncnn, cv2, numpy as np
# Cài trên RPi: pip install ncnn
# Copy: traffic_v10n.ncnn.param + traffic_v10n.ncnn.bin

net = ncnn.Net()
net.opt.num_threads = 4              # 4 core RPi 4
net.load_param("traffic_v10n.ncnn.param")
net.load_model("traffic_v10n.ncnn.bin")
# Output shape (300, 6) — NMS-free, giống ONNX
```

### TFLite INT8 — Edge AI / C++ (RPi / MCU)
```python
import tflite_runtime.interpreter as tflite
# Cài: pip install tflite-runtime   (nhẹ hơn tensorflow đầy đủ)

interp = tflite.Interpreter("traffic_v10n_int8.tflite", num_threads=4)
interp.allocate_tensors()
in_d  = interp.get_input_details()[0]
out_d = interp.get_output_details()[0]
# INT8: cần dequantize output: (tensor - zero_point) * scale
scale, zero = out_d["quantization"]

# Để deploy trên Coral Edge TPU:
# edgetpu_compiler traffic_v10n_int8.tflite
# → traffic_v10n_int8_edgetpu.tflite
```

### So sánh YOLOv8n vs YOLOv10n
| | YOLOv8n | YOLOv10n |
|--|---------|----------|
| Params | 3.2 M | 2.3 M (**−28%**) |
| FLOPs (320) | 8.7 G | 6.7 G (**−23%**) |
| NMS | Thủ công | **NMS-free** (~15% nhanh hơn) |
| Output ONNX | `(1, 4+nc, 8400)` | `(1, 300, 6)` — sẵn dùng |
| ultralytics | ≥8.2 | **≥8.3** |